# Potato Leaf Disease Detection — MobileNetV2 + TFLite
Uses Keras 3 standalone (TF 2.16+). Output: `mobilenet_model.tflite`

In [ ]:
import sys
print(sys.executable)

In [2]:
import os
import numpy as np
import tensorflow as tf

# TF 2.16+ uses standalone keras — import directly from keras
import keras
from keras.applications import MobileNetV2
from keras.layers import GlobalAveragePooling2D, Dense, Dropout
from keras.models import Model
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("TensorFlow:", tf.__version__)
print("Keras     :", keras.__version__)

os.chdir(r"C:\Users\Aaditya\OneDrive\Desktop\Projects\leaf-disease-detection")
print("Working dir:", os.getcwd())

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
DATASET    = "dataset"

# ── Data generators ───────────────────────────────────────────────
train_gen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.15
)

val_gen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_gen.flow_from_directory(
    DATASET, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', shuffle=True
)

val_data = val_gen.flow_from_directory(
    DATASET, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', shuffle=False
)

NUM_CLASSES = train_data.num_classes
print("\nClass indices:", train_data.class_indices)
print("Num classes  :", NUM_CLASSES)

ModuleNotFoundError: No module named 'tensorflow.python.trackable'

In [ ]:
# ── Build MobileNetV2 ─────────────────────────────────────────────
base = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base.trainable = False

x   = base.output
x   = GlobalAveragePooling2D()(x)
x   = Dense(128, activation='relu')(x)
x   = Dropout(0.3)(x)
out = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base.input, outputs=out)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Total params    : {model.count_params():,}")
print(f"Trainable params: {sum([w.numpy().size for w in model.trainable_weights]):,}")

In [ ]:
# ── Phase 1: Train head only (5 epochs) ──────────────────────────
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

print("=== Phase 1: head only ===")
history1 = model.fit(
    train_data, epochs=5,
    validation_data=val_data,
    callbacks=callbacks
)
print(f"Best val_accuracy: {max(history1.history['val_accuracy']):.4f}")

In [ ]:
# ── Phase 2: Fine-tune top 30 layers ─────────────────────────────
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("=== Phase 2: fine-tune top 30 layers ===")
history2 = model.fit(
    train_data, epochs=10,
    validation_data=val_data,
    callbacks=callbacks
)
print(f"Best val_accuracy: {max(history2.history['val_accuracy']):.4f}")

In [ ]:
# ── Evaluate ──────────────────────────────────────────────────────
loss, acc = model.evaluate(val_data)
print(f"\nFinal val accuracy : {acc*100:.2f}%")
print(f"Final val loss     : {loss:.4f}")
print(f"Class order used   : {train_data.class_indices}")
print()
print("CLASS_LABELS in app.py must match this order:")
sorted_classes = sorted(train_data.class_indices, key=train_data.class_indices.get)
for i, c in enumerate(sorted_classes):
    print(f"  {i} → {c}")

In [ ]:
# ── Convert to TFLite Float16 ─────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

with open('mobilenet_model.tflite', 'wb') as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / (1024 * 1024)
print(f"✅ Saved: mobilenet_model.tflite ({size_mb:.2f} MB)")

In [ ]:
# ── Sanity check: TFLite inference on a sample ────────────────────
from PIL import Image

CLASS_LABELS = ["Early Blight", "Late Blight", "Healthy"]

interp = tf.lite.Interpreter(model_path='mobilenet_model.tflite')
interp.allocate_tensors()
inp  = interp.get_input_details()
outp = interp.get_output_details()

print("Input shape :", inp[0]['shape'])   # (1, 224, 224, 3)
print("Output shape:", outp[0]['shape'])  # (1, 3)

TEST = "static/samples/late_blight/late1.jpg"
img  = Image.open(TEST).convert('RGB').resize((224, 224))
arr  = np.expand_dims(np.array(img, dtype=np.float32) / 255.0, axis=0)

interp.set_tensor(inp[0]['index'], arr)
interp.invoke()

preds = interp.get_tensor(outp[0]['index'])[0]
idx   = int(np.argmax(preds))

print(f"\nProbabilities: {preds}")
print(f"Predicted    : {CLASS_LABELS[idx]} ({preds[idx]*100:.2f}%)")
print("\n✅ TFLite model working correctly — ready to push")